# Label-Quality Audit — Inter-annotator Agreement with Confidence Intervals

**Self-contained analysis for the FIFA World Cup 2022 label-audit paper.**

Runs on the human-annotated workbook only (`FIFA_annotation_EASY_filled.xlsx`,
one tab per annotator) and reproduces:

* Fleiss' κ across the 5 annotators, with a 95% bootstrap CI
* mean pairwise Cohen's κ (human–human), with a 95% bootstrap CI and range
* the human-majority label (ties broken toward lower polarity) and its distribution
* Cohen's κ between the human-majority label and the CardiffNLP auto-label, with CI
* LaTeX-ready rows for the agreement table

**Requirements:** `pandas`, `numpy`, `openpyxl` only.

> The auto-label comparison needs the CardiffNLP label **for each of the 500
> audited tweets**. If `fifa_auto_labels.csv` holds exactly those 500 tweets
> (columns `id,auto_label`, produced by HOWTO Cell A) it is used; otherwise the
> notebook falls back to the study's reported confusion table. Both give κ = 0.126.
> A whole-corpus auto-label file (thousands of rows) is ignored automatically.

In [ ]:
import itertools, os
import numpy as np
import pandas as pd

# ----------------------- CONFIG -----------------------
ANNOTATION_FILE = 'FIFA_annotation_EASY_filled.xlsx'  # one tab per annotator
AUTO_LABELS_CSV = 'fifa_auto_labels.csv'              # optional: exactly the 500 tweets (id, auto_label)
B    = 10000        # bootstrap resamples
SEED = 42
rng  = np.random.default_rng(SEED)
LABELS = ['negative', 'neutral', 'positive']          # index 0,1,2

def to_idx(x):
    s = str(x).strip().lower()[:3]
    return {'neg': 0, 'neu': 1, 'pos': 2}.get(s, np.nan)

In [ ]:
# ---- load every annotator tab into an (items x annotators) matrix ----
xl   = pd.ExcelFile(ANNOTATION_FILE)
tabs = [t for t in xl.sheet_names if t.lower().startswith('annotator')]
assert len(tabs) >= 2, f'Expected annotator tabs, found: {xl.sheet_names}'

cols, ann_ids = [], None
for t in tabs:
    d = pd.read_excel(ANNOTATION_FILE, sheet_name=t)
    d.columns = [str(c).strip().lower() for c in d.columns]
    if 'id' in d.columns:
        d = d.sort_values('id').reset_index(drop=True)
        if ann_ids is None:
            ann_ids = d['id'].values
    sent_col = [c for c in d.columns if 'sent' in c or 'label' in c][0]
    cols.append(d[sent_col].map(to_idx).values)

ann = np.column_stack(cols).astype(float)   # (n_items, n_annotators)
n_items, n_ann = ann.shape
print(f'{n_items} tweets  x  {n_ann} annotators  ({tabs})')
print('annotation ids captured:', ann_ids is not None)
print('missing labels per annotator:', np.isnan(ann).sum(axis=0))

In [ ]:
# --------------------- kappa functions ---------------------
def cohen_kappa(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    ok = ~np.isnan(a) & ~np.isnan(b)
    a, b = a[ok], b[ok]
    po = np.mean(a == b)
    pe = sum(np.mean(a == c) * np.mean(b == c) for c in (0, 1, 2))
    return (po - pe) / (1 - pe) if (1 - pe) > 0 else np.nan

def _counts(a):
    mt = np.zeros((a.shape[0], 3))
    for i in range(a.shape[0]):
        for r in a[i]:
            if not np.isnan(r):
                mt[i, int(r)] += 1
    return mt

def fleiss_kappa(a):
    mt = _counts(a)
    n, _ = mt.shape
    N = mt.sum(axis=1)[0]
    p = mt.sum(axis=0) / (n * N)
    P = (np.sum(mt ** 2, axis=1) - N) / (N * (N - 1))
    return (P.mean() - np.sum(p ** 2)) / (1 - np.sum(p ** 2))

def mean_pairwise(a):
    ks = [cohen_kappa(a[:, i], a[:, j])
          for i, j in itertools.combinations(range(a.shape[1]), 2)]
    return float(np.mean(ks)), ks

def majority(row):
    v = [int(x) for x in row if not np.isnan(x)]
    c = [v.count(0), v.count(1), v.count(2)]
    m = max(c)
    return next(k for k in (0, 1, 2) if c[k] == m)   # tie -> lower polarity

def boot_ci(fn, *arrays, n_boot=B):
    n = len(arrays[0]); idx = np.arange(n); out = []
    for _ in range(n_boot):
        s = rng.choice(idx, n, replace=True)
        out.append(fn(*[np.asarray(x)[s] for x in arrays]))
    out = np.array([v for v in out if not np.isnan(v)])
    return float(np.percentile(out, 2.5)), float(np.percentile(out, 97.5))

In [ ]:
# --------------------- human-human agreement ---------------------
fk = fleiss_kappa(ann)
mpk, pairwise = mean_pairwise(ann)
fk_ci = boot_ci(lambda A: fleiss_kappa(A), ann)
mp_ci = boot_ci(lambda A: mean_pairwise(A)[0], ann)

print(f"Fleiss' kappa (5 annotators)      : {fk:.3f}  95% CI [{fk_ci[0]:.3f}, {fk_ci[1]:.3f}]")
print(f"Mean pairwise Cohen's kappa (h-h) : {mpk:.3f}  95% CI [{mp_ci[0]:.3f}, {mp_ci[1]:.3f}]")
print(f"  pairwise range                  : {min(pairwise):.3f} - {max(pairwise):.3f}")

In [ ]:
# --------------------- human-majority label + distribution ---------------------
maj  = np.array([majority(r) for r in ann])
dist = {LABELS[k]: int((maj == k).sum()) for k in (0, 1, 2)}
print('human-majority label distribution:', dist)

In [ ]:
# --------------------- human-majority vs. CardiffNLP auto-label ---------------------
auto = None
if os.path.exists(AUTO_LABELS_CSV):
    a = pd.read_csv(AUTO_LABELS_CSV)
    a.columns = [str(c).strip().lower() for c in a.columns]
    lab_col = [c for c in a.columns if 'auto' in c or 'label' in c or 'sent' in c][0]
    a['_lab'] = a[lab_col].map(to_idx)
    # Accept the CSV only if it holds EXACTLY the 500 audited tweets; a whole-corpus
    # file (thousands of rows) is rejected and we fall back to the confusion table.
    if len(a) == n_items:
        if 'id' in a.columns and ann_ids is not None and set(a['id']) == set(ann_ids.tolist()):
            m = pd.DataFrame({'id': ann_ids}).merge(a[['id', '_lab']].drop_duplicates('id'),
                                                    on='id', how='left')
            auto = m['_lab'].values
            print(f'Using per-tweet auto labels from {AUTO_LABELS_CSV} (aligned by id, {n_items} tweets).')
        else:
            auto = a['_lab'].values
            print(f'Using per-tweet auto labels from {AUTO_LABELS_CSV} (row-aligned, {n_items} rows).')
    else:
        print(f'{AUTO_LABELS_CSV} has {len(a)} rows, not the {n_items} audited tweets '
              f'-> ignoring it and using the reported confusion table.')
    if auto is not None and np.isnan(np.asarray(auto, float)).any():
        print('  (some auto labels missing after alignment -> using confusion table instead.)')
        auto = None

if auto is not None:
    h_pairs, a_pairs = maj.astype(float), auto.astype(float)
else:
    # study's reported human-majority (rows) x auto (cols) confusion table; order neg,neu,pos
    AUTO_CONFUSION = np.array([[ 18,   6,   4],
                               [100, 154, 136],
                               [ 15,  20,  47]])
    h_pairs, a_pairs = [], []
    for i in range(3):
        for j in range(3):
            h_pairs += [i] * AUTO_CONFUSION[i, j]
            a_pairs += [j] * AUTO_CONFUSION[i, j]
    h_pairs, a_pairs = np.array(h_pairs, float), np.array(a_pairs, float)
    print('Using reported human-vs-auto confusion table.')

k_ha  = cohen_kappa(h_pairs, a_pairs)
ha_ci = boot_ci(cohen_kappa, h_pairs, a_pairs)
raw   = float(np.mean(h_pairs == a_pairs))
print(f"Human-majority vs. auto-label     : {k_ha:.3f}  95% CI [{ha_ci[0]:.3f}, {ha_ci[1]:.3f}]  raw={raw*100:.1f}%")

In [ ]:
# --------------------- LaTeX-ready rows for the agreement table ---------------------
# backslash built via chr(92) to keep the source escaping-free
bs = chr(92); kap = bs + 'kappa'; er = bs + bs
print(f"Fleiss' ${kap}$ (5 annotators) & {fk:.3f} & [{fk_ci[0]:.3f}, {fk_ci[1]:.3f}] & Moderate {er}")
print(f"Mean pairwise Cohen's ${kap}$ & {mpk:.3f} & [{mp_ci[0]:.3f}, {mp_ci[1]:.3f}] & Moderate {er}")
print(f"Pairwise range & {min(pairwise):.3f}--{max(pairwise):.3f} & --- & Moderate {er}")
print(f"Human majority vs.{bs} auto-label & {k_ha:.3f} & [{ha_ci[0]:.3f}, {ha_ci[1]:.3f}] & Slight {er}")

## Optional: model-vs-human-gold table

This is the one analysis the human file alone **cannot** produce — it needs each
model's predicted label on the same 500 tweets. Export `model_preds_on_500.csv`
with a `tweet_id` column plus one column per model (labels in {negative, neutral,
positive} or {0,1,2}) — see `HOWTO_model_vs_human_gold.md`. The cell below aligns
the predictions to the annotation ids and reports each model's accuracy and
Cohen's κ against the human-majority label — “which model best matches humans?”.

In [ ]:
PRED_CSV = 'model_preds_on_500.csv'
bs = chr(92); kap = bs + 'kappa'; er = bs + bs; pct = '%'
if os.path.exists(PRED_CSV):
    pred = pd.read_csv(PRED_CSV)
    pred.columns = [str(c).strip() for c in pred.columns]
    key = next((c for c in pred.columns if str(c).lower() in ('tweet_id', 'id', 'tweetid')), None)
    if key is not None and ann_ids is not None:      # align predictions to annotation ids
        pred = pd.DataFrame({key: ann_ids}).merge(pred, on=key, how='left')
    model_cols = [c for c in pred.columns if c != key]
    gold = maj.astype(float)
    print(f"Model & Acc. vs human & Cohen's ${kap}$ vs human (95{pct} CI) {er}")
    rows = []
    for mname in model_cols:
        mp = pred[mname].map(to_idx).values
        ok = ~np.isnan(mp) & ~np.isnan(gold)
        acc = float(np.mean(mp[ok] == gold[ok]))
        kk  = cohen_kappa(gold[ok], mp[ok])
        kci = boot_ci(cohen_kappa, gold[ok], mp[ok])
        rows.append((mname, acc, kk))
        print(f"{mname} & {acc*100:.1f}{pct} & {kk:.3f} [{kci[0]:.3f}, {kci[1]:.3f}] {er}")
    best = max(rows, key=lambda r: r[2])
    print(f"Best human-agreement model: {best[0]} (kappa={best[2]:.3f})")
else:
    print(f"No {PRED_CSV} found — add model predictions on the 500 tweets to build this table.")